In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

In [62]:
N_users = 10000
N_times_train = 10
N_times_val = 5
N_times_prod = 5

# Put an average of 2 users in each category. Not perfect memorization.
N_categories = N_users // 2
# Bounds for the default probabilities
p_min = 0.01
p_max = 0.20

# Inverse regularization
# Set this very high so that we do get overfitting
C = 1e6

seed = 1235

In [63]:
# Transformations of parameters, do not change these

# Convert min and max probabilities into logits
z_max = np.log(p_max/(1-p_max))
z_min = np.log(p_min/(1-p_min))

In [64]:
# Generate the dataset
rng = np.random.default_rng(seed)

dataset = []  # (t, user_idx, x0, x1, y)

for idx_user in range(N_users):
    #cat_user = rng.choice(N_categories)
    cat_user = idx_user
    # Run a simulation forward in time to determine when the user defaults (if ever)
    for t in range(N_times_train + N_times_val + N_times_prod):
        # Reset user circumstances each time
        z_user = rng.uniform(low=z_min, high=z_max)
        p_user = 1 / (1 + np.exp(-z_user))
        
        does_user_default = rng.binomial(p=p_user, n=1)
        dataset.append((t, idx_user, z_user, cat_user, does_user_default))
        if does_user_default:
            break

# Make it a pandas DataFrame for easier manipulation
dataset_pd = pd.DataFrame(
    dataset,
    columns=['t', 'user_idx', 'x0', 'x1', 'y']
)

feature_cols = ['x0', 'x1']

In [65]:
# Fit a simple model to the training data and show that the regression coefficient is 1.0
lr_simple_model = LogisticRegression(C=C)
lr_simple_model.fit(
    X = dataset_pd[dataset_pd['t'] < N_times_train][['x0']],
    y = dataset_pd[dataset_pd['t'] < N_times_train]['y'],
)

print(f'Regression coefficient of simple model: {lr_simple_model.coef_[0, 0]}')

Regression coefficient of simple model: 1.0156165533512984


In [70]:
def run_experiment(
    dataset_pd: pd.DataFrame,
    train_val_scheme: str,
):
    # This section gets affected by our choice of val scheme
    df_train = dataset_pd[dataset_pd['t'] < N_times_train]
    df_val = dataset_pd[(dataset_pd['t'] >= N_times_train) & (dataset_pd['t'] < N_times_train + N_times_val)]

    if train_val_scheme == 'battenburg':
        df_train = df_train[df_train['user_idx'] < (N_users // 2)]
        df_val = df_val[df_val['user_idx'] >= (N_users // 2)]
    elif train_val_scheme == 'half-population out-of-time':
        df_train = df_train[df_train['user_idx'] < (N_users // 2)]
        df_val = df_val[df_val['user_idx'] < (N_users // 2)]
    elif train_val_scheme == 'out-of-time':
        pass
    else:
        raise ValueError(f'Unrecognized train_val_scheme: {train_val_scheme}')
    
    # Everything below here does not
    X_train = df_train[feature_cols]
    y_train = df_train['y']
    
    X_val = df_val[feature_cols]
    y_val = df_val['y']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('none', 'passthrough', ['x0']),
            ('one-hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['x1'])
        ]
    )
    
    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', LogisticRegression(C=C, tol=1e-3))
        ]
    )
    
    pipeline.fit(X_train, y_train)

    y_score_train = pipeline.predict_proba(X_train)[:, 1]
    roc_auc_train = roc_auc_score(y_true=y_train, y_score=y_score_train)
    pr_auc_train = average_precision_score(y_true=y_train, y_score=y_score_train)
    log_loss_train = log_loss(y_true=y_train, y_pred=y_score_train)
    avg_p_for_pos_train = y_score_train @ y_train / np.sum(y_train)

    y_score_val = pipeline.predict_proba(X_val)[:, 1]
    roc_auc_val = roc_auc_score(y_true=y_val, y_score=y_score_val)
    pr_auc_val = average_precision_score(y_true=y_val, y_score=y_score_val)
    log_loss_val = log_loss(y_true=y_val, y_pred=y_score_val)
    avg_p_for_pos_val = y_score_val @ y_val / np.sum(y_val)
    
    # Retrain for prod
    df_retrain = dataset_pd[dataset_pd['t'] < N_times_train + N_times_val]
    df_prod = dataset_pd[dataset_pd['t'] >= N_times_train + N_times_val]

    X_retrain = df_retrain[feature_cols]
    y_retrain = df_retrain['y']
    
    X_prod = df_prod[feature_cols]
    y_prod = df_prod['y']
    
    pipeline.fit(X_retrain, y_retrain)

    y_score_retrain = pipeline.predict_proba(X_retrain)[:, 1]
    y_score_prod = pipeline.predict_proba(X_prod)[:, 1]

    # Compute each metric for each period
    df_metrics = pd.DataFrame(
        [
            {
                'dataset': ds_name,
                'roc_auc': roc_auc_score(y_true=y_this_ds, y_score=y_score_this_ds),
                'pr_auc': average_precision_score(y_true=y_this_ds, y_score=y_score_this_ds),
                'log_loss': log_loss(y_true=y_this_ds, y_pred=y_score_this_ds),
                'avg_p_for_pos': y_score_this_ds @ y_this_ds / np.sum(y_this_ds),
            }
             for ds_name, y_this_ds, y_score_this_ds in [
                 ('train', y_train, y_score_train),
                 ('retrain', y_retrain, y_score_retrain),
                 ('val', y_val, y_score_val),
                 ('prod', y_prod, y_score_prod),
             ]
        ]
    )
    
    return df_metrics

In [71]:
run_experiment(dataset_pd, train_val_scheme='out-of-time')

,dataset,roc_auc,pr_auc,log_loss,avg_p_for_pos
0,train,0.947687,0.590394,0.128740,0.363542
1,retrain,0.920149,0.508551,0.149432,0.316869
2,val,0.703872,0.135128,0.381370,0.006317
3,prod,0.711455,0.149647,0.433989,0.003924


In [72]:
run_experiment(dataset_pd, train_val_scheme='battenburg')

,dataset,roc_auc,pr_auc,log_loss,avg_p_for_pos
0,train,0.949512,0.600425,0.124702,0.400733
1,retrain,0.920149,0.508551,0.149432,0.316869
2,val,0.736538,0.152468,0.283145,0.030049
3,prod,0.711455,0.149647,0.433989,0.003924


In [73]:
run_experiment(dataset_pd, train_val_scheme='half-population out-of-time')

,dataset,roc_auc,pr_auc,log_loss,avg_p_for_pos
0,train,0.949512,0.600425,0.124702,0.400733
1,retrain,0.920149,0.508551,0.149432,0.316869
2,val,0.687322,0.128464,0.469747,0.001912
3,prod,0.711455,0.149647,0.433989,0.003924


In [29]:
run_experiment(dataset_pd, battenburg_validation=True)

,dataset,roc_auc,pr_auc,log_loss,avg_p_for_pos
0,train,0.866669,0.369494,0.179267,0.240001
1,retrain,0.803648,0.262003,0.204328,0.168658
2,val,0.646724,0.108512,0.292747,0.161125
3,prod,0.652305,0.117768,0.262726,0.073486


In [187]:
dataset_pd

,t,user_idx,x0,x1,y
0,0,0,-4.006474,1878,0
1,1,0,-1.905789,1878,0
2,2,0,-3.654247,1878,0
3,3,0,-2.691103,1878,0
4,4,0,-2.510687,1878,0
...,...,...,...,...,...
113162,7,9999,-3.830854,2958,0
113163,8,9999,-2.462617,2958,0
113164,9,9999,-2.483117,2958,0
113165,10,9999,-4.095047,2958,0


In [179]:
pipeline.steps[1][1].coef_

array([[ 1.07672272,  0.47831347, -0.40957374, ..., -0.478598  ,
        -0.09502617,  0.06142921]])